In [1]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import faiss
import pickle

C:\Users\atliu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:
row_data = pd.read_csv("fra_perfumes.csv")

In [30]:
row_data.head()

,Name,Gender,Rating Value,Rating Count,Main Accords,Perfumers,Description,url
0,9am Afnanfor women,for women,3.73,174,"['citrus', 'musky', 'woody', 'aromatic', 'warm...",[],9ambyAfnanis a fragrance for women. Top notes ...,https://www.fragrantica.com/perfume/Afnan/9am-...
1,9am Dive Afnanfor women and men,for women and men,4.29,842,"['fruity', 'woody', 'green', 'warm spicy', 'ar...",[],9am DivebyAfnanis a Aromatic Aquatic fragrance...,https://www.fragrantica.com/perfume/Afnan/9am-...
2,9am pour Femme Afnanfor women,for women,4.00,68,"['fruity', 'musky', 'amber', 'citrus', 'powder...",[],9am pour FemmebyAfnanis a Amber fragrance for ...,https://www.fragrantica.com/perfume/Afnan/9am-...
3,9pm Afnanfor men,for men,4.50,"6,865","['vanilla', 'amber', 'warm spicy', 'cinnamon',...",[],9pmbyAfnanis a Amber Vanilla fragrance for men...,https://www.fragrantica.com/perfume/Afnan/9pm-...
4,9pm pour Femme Afnanfor women,for women,3.49,63,"['woody', 'aromatic', 'rose', 'fruity', 'powde...",[],9pm pour FemmebyAfnanis a Amber Floral fragran...,https://www.fragrantica.com/perfume/Afnan/9pm-...


In [31]:
row_data["Main Accords"][10]

"['woody', 'powdery', 'musky', 'amber', 'patchouli', 'vanilla', 'green', 'warm spicy', 'balsamic', 'animalic']"

# Loading Embeddings

In [2]:
with open('data.pkl', 'rb') as f:
   data = pickle.load(f)

In [3]:
desc_embs = np.load("desc_embs.npy")
accord_embs = np.load("accord_embs.npy")
gender_embs = np.load("gender_embss.npy")
brand_embs = np.load("brand_embs.npy")

# Generating Final Embeddings

In [4]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [5]:
W_DESC   = 0.35
W_ACCORD = 0.25
W_GENDER = 0.30
W_BRAND = 0.1

final_embs = (
    W_DESC   * desc_embs +
    W_ACCORD * accord_embs +
    W_GENDER * gender_embs + 
    W_BRAND  * brand_embs
)

final_embs = final_embs / np.linalg.norm(final_embs, axis=1, keepdims=True)

dim = final_embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(final_embs.astype(np.float32))

In [6]:
brands = sorted(data["Brand"].unique())
brand_emb_dict = {}

for b in brands:
    # find any row belonging to this brand
    idx = data.index[data["Brand"] == b][0]
    brand_emb_dict[b] = brand_embs[idx] / np.linalg.norm(brand_embs[idx])

mean_brand_emb = brand_embs.mean(axis=0)
mean_brand_emb = mean_brand_emb / np.linalg.norm(mean_brand_emb)

# Query

In [59]:
def expand_gender_vec(gv, dim=384):
    # Convert 3-dim gender vector → same dimensionality as BERT embedding
    return np.pad(gv, (0, dim - len(gv)), 'constant')

def detect_brand_in_query(text, brands):
    """ Detect if any brand from the list is mentioned in the text.
        Return the brand if found, else None."""
    t = text.lower()
    for b in brands:
        if b.lower() in t:
            return b
    return None


def detect_gender_in_query(text):
    """Detech gender from the text.
       Return a 2D numpy array representing the gender embedding."""
    t = text.lower()
    if "unisex" in t:
        return np.array([1/np.sqrt(2), 1/np.sqrt(2)])
    if "men" in t or "male" in t:
        return np.array([1.0, 0.0])
    if "women" in t or "female" in t:
        return np.array([0.0, 1.0])
    # default neutral
    return np.array([1/np.sqrt(2), 1/np.sqrt(2)])

def clean_query_text(text, brands):
    """Remove known brand names & gender words so BERT focuses on scent description."""
    t = text.lower()
    for b in brands:
        t = t.replace(b.lower(), "")
    t = t.replace("unisex", "")
    t = t.replace("men", "")
    t = t.replace("man", "")
    t = t.replace("male", "")
    t = t.replace("women", "")
    t = t.replace("woman", "")
    t = t.replace("female", "")
    return t.strip()


def clean_query_text(text, brands):
    """Remove known brand names & gender words so BERT focuses on scent description."""
    t = text.lower()
    for b in brands:
        t = t.replace(b.lower(), "")
    t = t.replace("unisex", "")
    t = t.replace("men", "")
    t = t.replace("man", "")
    t = t.replace("male", "")
    t = t.replace("women", "")
    t = t.replace("woman", "")
    t = t.replace("female", "")
    return t.strip()

def embed_query(text, model, brands, mean_brand_emb):
    # ----- BRAND -----
    brand = detect_brand_in_query(text, brands)
    if brand:
        brand_emb = brand_emb_dict[brand]     # PCA-reduced or SBERT brand embedding
    else:
        brand_emb = mean_brand_emb            # neutral brand

    # ----- GENDER -----
    gender_vec = detect_gender_in_query(text)
    gender_emb = expand_gender_vec(gender_vec)   # your 2-dim → 384-dim function

    # ----- CLEAN TEXT FOR DESCRIPTION + ACCORD -----
    cleaned = clean_query_text(text, brands)

    # ----- SBERT DESCRIPTION + ACCORD ENCODING -----
    desc_emb   = model.encode([cleaned])[0]
    accord_emb = model.encode([cleaned])[0]

    # ----- FUSION -----
    q = (
        W_DESC   * desc_emb +
        W_ACCORD * accord_emb +
        W_GENDER * gender_emb +
        W_BRAND  * brand_emb
    )

    q = q / np.linalg.norm(q)
    return q.astype(np.float32)

def recommend(query, model=model, brands=brands, mean_brand_emb=mean_brand_emb, faiss_index=index, data=data, top_k=10):
    q_emb = embed_query(query, model, brands, mean_brand_emb)
    
    D, I = faiss_index.search(q_emb.reshape(1, -1), top_k)
    
    results = []
    for idx, dist in zip(I[0], D[0]):
        row = data.iloc[idx]
        results.append({
            "Name": row["Name"],
            "Brand": row["Brand"],
            "Gender": row["Gender"],
            "Main Accords": row["Main Accords"],
            "Description": row["Description"],
            "Main Accords": row_data["Main Accords"][idx],
            "Score": float(dist)
        })
    results_df = pd.DataFrame(results)
    results_df["Main Accords"] = results_df["Main Accords"].apply(lambda x: x.replace("[]", "Not Specified"))
    results_df["Main Accords"] = results_df["Main Accords"].apply(lambda x: x.replace("]", "").replace("[", "").replace("'", "").replace(",", ", "))
    return results_df, results

In [60]:
out, _ = recommend(query="fresh citrus green summer unisex dior")

In [61]:
out

,Name,Brand,Gender,Main Accords,Description,Score
0,Oud Caravan No 3 Abdes Salaam Attars,Perfumes,unisex,"oud, fresh spicy",Oud Caravan No 3byAbdes Salaam Attars Perfumes...,0.516041
1,Vetyver Bastide Aix en,Provence,unisex,Not Specified,VetyverbyBastide Aix en Provenceis a Woody fra...,0.487143
2,#455 Tabac écarlate Le Ré,Noir,unisex,Not Specified,#455 Tabac écarlatebyLe Ré Noiris a Amber Spic...,0.483210
3,Aquilae - Oud du Désert Diane,Castel,unisex,"oud, fresh spicy",Aquilae - Oud du DésertbyDiane Castelis a frag...,0.473504
4,Esme L.T.,Piver,unisex,Not Specified,EsmebyL.T. Piveris a fragrance for women and m...,0.461535
5,Illusive Thorn The Dua,Brand,unisex,"oud, fresh spicy",Illusive ThornbyThe Dua Brandis a Amber fragra...,0.457381
6,Oud Té Fueguia,1833,unisex,"oud, green, fresh, fresh spicy",Oud TébyFueguia 1833is a fragrance for women a...,0.456458
7,EO Black Ensar,Oud,unisex,"oud, fresh spicy",EO BlackbyEnsar Oudis a fragrance for women an...,0.454804
8,#211 Mon Vetiver Narcotique Le Ré,Noir,unisex,"aromatic, fruity, green, woody, cannabis, ...",#211 Mon Vetiver NarcotiquebyLe Ré Noiris a fr...,0.454687
9,Oud Caravan No 2 Abdes Salaam Attars,Perfumes,unisex,"oud, white floral, woody, animalic",Oud Caravan No 2byAbdes Salaam Attars Perfumes...,0.454087
